In [1]:
from pyspark.sql.functions import year, count, round as spark_round, col

StatementMeta(, f8c85d3a-9aa7-4042-bbb8-1e416ad74380, 3, Finished, Available, Finished, False)

In [2]:
# nb_tasa_incidencias
# Calcula tasa de incidencias por zona y año

df_incidencias = spark.read.table("lh_transandino.dbo.silver_incidencias")
df_envios      = spark.read.table("lh_transandino.dbo.silver_envios")
df_transportistas = spark.read.table("lh_transandino.dbo.silver_transportistas")

from pyspark.sql.functions import year, count, round as spark_round

# Join incidencias → envios → transportistas para obtener zona
df = (
    df_incidencias
    .join(df_envios.select("id_envio", "id_transportista", "fecha_envio"),
          on="id_envio", how="left")
    .join(df_transportistas.select("id_transportista", "zona"),
          on="id_transportista", how="left")
    .withColumn("anio", year("fecha_envio"))
)

# Total envios por zona y año
df_total = (
    df_envios
    .join(df_transportistas.select("id_transportista", "zona"),
          on="id_transportista", how="left")
    .withColumn("anio", year("fecha_envio"))
    .groupBy("zona", "anio")
    .agg(count("id_envio").alias("total_envios"))
)

# Total incidencias por zona y año
df_inc_zona = (
    df.groupBy("zona", "anio")
    .agg(count("id_incidencia").alias("total_incidencias"))
)

# Join y calcular tasa
df_tasa = (
    df_inc_zona
    .join(df_total, on=["zona", "anio"], how="left")
    .withColumn(
        "tasa_incidencias_pct",
        spark_round(
            (col("total_incidencias") / col("total_envios")) * 100, 2
        )
    )
    .orderBy("zona", "anio")
)

df_tasa.show()

# Guardar como tabla Delta nativa en el Lakehouse
df_tasa.write.format("delta").mode("overwrite").saveAsTable("lh_transandino.dbo.gold_tasa_incidencias")
print("Tabla gold_tasa_incidencias creada exitosamente.")

StatementMeta(, f8c85d3a-9aa7-4042-bbb8-1e416ad74380, 5, Finished, Available, Finished, False)

+------+----+-----------------+------------+--------------------+
|  zona|anio|total_incidencias|total_envios|tasa_incidencias_pct|
+------+----+-----------------+------------+--------------------+
|Centro|2022|                6|          21|               28.57|
|Centro|2023|                9|          44|               20.45|
|Centro|2024|                8|          45|               17.78|
|  Este|2022|                5|          30|               16.67|
|  Este|2023|               14|          41|               34.15|
|  Este|2024|               17|          62|               27.42|
| Norte|2022|                5|          26|               19.23|
| Norte|2023|               15|          44|               34.09|
| Norte|2024|               18|          48|                37.5|
| Oeste|2022|                8|          27|               29.63|
| Oeste|2023|               15|          53|                28.3|
| Oeste|2024|               13|          44|               29.55|
|   Sur|20